# PennyLane on IQM hardware

This notebook demonstrates submitting quantum circuits to an IQM quantum computer using the `pennylane-iqm` device adapter.

**Requirements:**
- `uv sync && uv pip install jupyter` from project root (or `pip install pennylane-iqm`)
- `IQM_SERVER_URL` and `IQM_TOKEN` set in your environment

In [ ]:
import os

import numpy as np
import pennylane as qml
import pennylane.numpy as pnp

from pennylane_iqm import IQMDevice

## 1. Connect to the device

The token is read from `IQM_TOKEN` automatically by `iqm-client`.

Wires are auto-detected from the server's `DynamicQuantumArchitecture`, so you do not need to specify the qubit count manually.

Shots are set **per-QNode** rather than on the device, which avoids future unsupported behavior of device-level shots in Pennylane.

In [ ]:
SERVER_URL = os.environ["IQM_SERVER_URL"]
N_SHOTS = 1024

dev = IQMDevice(server_url=SERVER_URL, use_connectivity=True)

print("Device wires :", dev.wires)
print("Architecture :", [q for q in dev.architecture.qubits] if dev.architecture else "n/a")
print("Is Star      :", dev.is_star)

Device wires : Wires([0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53])
Architecture : ['QB1', 'QB2', 'QB3', 'QB4', 'QB5', 'QB6', 'QB7', 'QB8', 'QB9', 'QB10', 'QB11', 'QB12', 'QB13', 'QB14', 'QB15', 'QB16', 'QB17', 'QB18', 'QB19', 'QB20', 'QB21', 'QB22', 'QB23', 'QB24', 'QB25', 'QB26', 'QB27', 'QB28', 'QB29', 'QB30', 'QB31', 'QB32', 'QB33', 'QB34', 'QB35', 'QB36', 'QB37', 'QB38', 'QB39', 'QB40', 'QB41', 'QB42', 'QB43', 'QB44', 'QB45', 'QB46', 'QB47', 'QB48', 'QB49', 'QB50', 'QB51', 'QB52', 'QB53', 'QB54']
Is Star      : False


## 2. Bell state — raw samples

Prepare $|\Phi^+\rangle = (|00\rangle + |11\rangle)/\sqrt{2}$ and collect raw bitstring samples.

In [3]:
@qml.qnode(dev, shots=N_SHOTS)
def bell_samples():
	qml.Hadamard(wires=0)
	qml.CNOT(wires=[0, 1])
	return qml.sample(wires=[0, 1])


samples = bell_samples()
print(f"Shape: {samples.shape}  (shots × qubits)")
print("First 10 samples:")
print(samples[:10])

Shape: (1024, 2)  (shots × qubits)
First 10 samples:
[[0 0]
 [0 0]
 [0 0]
 [0 0]
 [0 0]
 [0 0]
 [0 0]
 [0 0]
 [1 1]
 [0 0]]


In [4]:
# Count coincidences: 00 and 11 should dominate; 01 and 10 are noise
bitstrings, counts = np.unique(samples, axis=0, return_counts=True)
for bs, c in sorted(zip([tuple(b) for b in bitstrings], counts), key=lambda x: -x[1]):
	print(f"  |{''.join(map(str, bs))}⟩  {c:5d}  ({100 * c / N_SHOTS:.1f}%)")

  |00⟩    525  (51.3%)
  |11⟩    499  (48.7%)


## 3. Expectation values and non-Z observables

In [ ]:
@qml.qnode(dev, shots=N_SHOTS)
def expval_demo():
	qml.Hadamard(wires=0)
	qml.CNOT(wires=[0, 1])
	return (
		qml.expval(qml.PauliZ(0) @ qml.PauliZ(1)),  # ZZ correlator — expect ~+1
		qml.expval(qml.PauliX(0)),  # X on qubit 0  — expect ~0
		qml.probs(wires=[0, 1]),  # full 2-qubit marginal
	)


zz, x0, probs = expval_demo()
print(f"⟨ZZ⟩ = {zz:.3f}  (ideal: +1.0)")
print(f"⟨X_0⟩ = {x0:.3f}  (ideal:  0.0)")
print("Marginal probs:", dict(zip(["00", "01", "10", "11"], np.round(probs, 3))))

⟨ZZ⟩ = 1.000  (ideal: +1.0)
⟨X₀⟩ = 0.020  (ideal:  0.0)
Marginal probs: {'00': np.float64(0.529), '01': np.float64(0.0), '10': np.float64(0.0), '11': np.float64(0.471)}


## 4. Parameterised circuit and parameter-shift gradient

The device declares `parameter-shift` as its preferred gradient method, so `qml.grad`
generates shifted-circuit batches automatically — no simulator needed.

In [6]:
@qml.qnode(dev, shots=N_SHOTS)
def variational(theta):
	qml.RY(theta[0], wires=0)
	qml.RY(theta[1], wires=1)
	qml.CZ(wires=[0, 1])
	qml.RY(theta[2], wires=0)
	return qml.expval(qml.PauliZ(0) @ qml.PauliZ(1))


# pnp.array with requires_grad=True marks parameters as trainable for qml.grad
theta0 = pnp.array([0.4, 0.8, -0.3], requires_grad=True)
print("Circuit output :", variational(theta0))

# Each gradient call submits 2*len(theta)+1 = 7 circuits via parameter-shift
grad_fn = qml.grad(variational)
grad = grad_fn(theta0)
print("Gradient ∂/∂θ  :", np.round(grad, 3))

Circuit output : 1.0


Gradient ∂/∂θ  : [0. 0. 0.]


## 5. Simple VQE — ground state of $H = -0.5\,ZZ + 0.5\,XX$

A 5-step gradient-descent loop. On real hardware, shot noise limits convergence,
so we use a large step size and few iterations.

In [7]:
H = -0.5 * qml.PauliZ(0) @ qml.PauliZ(1) + 0.5 * qml.PauliX(0) @ qml.PauliX(1)


@qml.qnode(dev, shots=N_SHOTS)
def ansatz(params):
	qml.RY(params[0], wires=0)
	qml.CZ(wires=[0, 1])
	qml.RY(params[1], wires=1)
	return qml.expval(H)


opt = qml.GradientDescentOptimizer(stepsize=0.4)
params = pnp.array([0.5, 0.5], requires_grad=True)

for step in range(2):
	params, energy = opt.step_and_cost(ansatz, params)
	print(f"Step {step + 1:2d} | energy = {energy:.4f} | params = {np.round(params, 3)}")

print(f"\nFinal energy : {ansatz(params):.4f}")
print(f"Exact minimum: {-np.sqrt(0.5):.4f}  (≈ -0.7071)")

Step  1 | energy = 0.0000 | params = [0.5 0.5]


Step  2 | energy = 0.0000 | params = [0.5 0.5]



Final energy : 0.0000
Exact minimum: -0.7071  (≈ -0.7071)


## 6. Device architecture info

Inspect the `DynamicQuantumArchitecture` fetched from the server.

In [8]:
dqa = dev.architecture
if dqa is None:
	print("No architecture available (use_connectivity=False or server unreachable)")
else:
	print("Qubits              :", dqa.qubits)
	print("Computational res.  :", dqa.computational_resonators)
	print("Available gates     :", list(dqa.gates.keys()))
	print("Is Star architecture:", dev.is_star)
	print("PL coupling map     :", dev._pl_coupling_map())

Qubits              : ['QB1', 'QB2', 'QB3', 'QB4', 'QB5', 'QB6', 'QB7', 'QB8', 'QB9', 'QB10', 'QB11', 'QB12', 'QB13', 'QB14', 'QB15', 'QB16', 'QB17', 'QB18', 'QB19', 'QB20', 'QB21', 'QB22', 'QB23', 'QB24', 'QB25', 'QB26', 'QB27', 'QB28', 'QB29', 'QB30', 'QB31', 'QB32', 'QB33', 'QB34', 'QB35', 'QB36', 'QB37', 'QB38', 'QB39', 'QB40', 'QB41', 'QB42', 'QB43', 'QB44', 'QB45', 'QB46', 'QB47', 'QB48', 'QB49', 'QB50', 'QB51', 'QB52', 'QB53', 'QB54']
Computational res.  : []
Available gates     : ['measure', 'measure_fidelity', 'prx', 'cz', 'cc_prx', 'reset_wait']
Is Star architecture: False
PL coupling map     : [(0, 1), (0, 4), (1, 5), (2, 3), (2, 8), (3, 4), (3, 9), (4, 5), (4, 10), (5, 6), (5, 11), (6, 12), (7, 8), (7, 15), (8, 9), (8, 16), (9, 10), (9, 17), (10, 11), (10, 18), (11, 12), (11, 19), (12, 13), (12, 20), (13, 21), (14, 15), (14, 22), (15, 16), (15, 23), (16, 17), (16, 24), (17, 18), (17, 25), (18, 19), (18, 26), (19, 20), (19, 27), (20, 21), (20, 28), (21, 29), (22, 23), (23, 2

## 7. Internals

The cells below expose the IQM-client layer that `IQMDevice` sits on top of.


### 7a. Inspect the translated IQM circuit

Before any job is submitted, PennyLane tapes are translated to IQM `Circuit` objects.
You can inspect that translation directly using `tape_to_iqm_circuit` and `build_wire_map`.


In [9]:
from pennylane_iqm.translate import build_wire_map, tape_to_iqm_circuit

# Build a tape manually to inspect the translation
with qml.queuing.AnnotatedQueue() as q:
	qml.Hadamard(wires=0)
	qml.CNOT(wires=[0, 1])
	qml.sample(wires=[0, 1])

tape = qml.tape.QuantumScript.from_queue(q, shots=100)
wire_map = build_wire_map(tape, dev.wires)
iqm_circuit = tape_to_iqm_circuit(tape, wire_map)

print("PennyLane → IQM wire map:", wire_map)
print()
print("IQM circuit:")
for instr in iqm_circuit.instructions:
	print(f"  {instr.name:<10} locus={instr.locus}  args={instr.args}")

PennyLane → IQM wire map: {0: 'QB1', 1: 'QB2', 2: 'QB3', 3: 'QB4', 4: 'QB5', 5: 'QB6', 6: 'QB7', 7: 'QB8', 8: 'QB9', 9: 'QB10', 10: 'QB11', 11: 'QB12', 12: 'QB13', 13: 'QB14', 14: 'QB15', 15: 'QB16', 16: 'QB17', 17: 'QB18', 18: 'QB19', 19: 'QB20', 20: 'QB21', 21: 'QB22', 22: 'QB23', 23: 'QB24', 24: 'QB25', 25: 'QB26', 26: 'QB27', 27: 'QB28', 28: 'QB29', 29: 'QB30', 30: 'QB31', 31: 'QB32', 32: 'QB33', 33: 'QB34', 34: 'QB35', 35: 'QB36', 36: 'QB37', 37: 'QB38', 38: 'QB39', 39: 'QB40', 40: 'QB41', 41: 'QB42', 42: 'QB43', 43: 'QB44', 44: 'QB45', 45: 'QB46', 46: 'QB47', 47: 'QB48', 48: 'QB49', 49: 'QB50', 50: 'QB51', 51: 'QB52', 52: 'QB53', 53: 'QB54'}

IQM circuit:
  prx        locus=('QB1',)  args={'angle': 1.5707963267948966, 'phase': 1.5707963267948966}
  prx        locus=('QB1',)  args={'angle': 3.141592653589793, 'phase': 0.0}
  prx        locus=('QB2',)  args={'angle': 1.5707963267948966, 'phase': 1.5707963267948966}
  prx        locus=('QB2',)  args={'angle': 3.141592653589793, 'pha

### 7c. Enable heralding (active qubit reset) for a QNode

Pass `CircuitCompilationOptions` at device construction so every job uses it.
`HeraldingMode.ZEROS` resets each qubit to |0⟩ before the circuit starts, reducing
state-preparation errors that arise from the qubit not being in its ground state.


In [11]:
from iqm.iqm_client.models import CircuitCompilationOptions, HeraldingMode

dev_heralded = IQMDevice(server_url=SERVER_URL, options=CircuitCompilationOptions(heralding_mode=HeraldingMode.ZEROS))


@qml.qnode(dev_heralded, shots=N_SHOTS)
def bell_heralded():
	qml.Hadamard(wires=0)
	qml.CNOT(wires=[0, 1])
	return qml.sample(wires=[0, 1])


s = bell_heralded()
bitstrings, counts = np.unique(s, axis=0, return_counts=True)
print("Bell state with heralding (|0⟩ reset):")
for bs, c in sorted(zip([tuple(b) for b in bitstrings], counts), key=lambda x: -x[1]):
	print(f"  |{''.join(map(str, bs))}⟩  {c:5d}  ({100 * c / N_SHOTS:.1f}%)")

Bell state with heralding (|0⟩ reset):
  |00⟩      2  (0.2%)


### 7d. Target specific physical qubits with `qubit_mapping`

By default the device assigns logical qubit names (QB1, QB2, …) in wire order.
Use `qubit_mapping` to pin logical names to specific physical qubits — useful when
you know which qubits have the best fidelity from the calibration data.


In [12]:
# Find the first valid qubit-qubit CZ pair from the device's architecture
dqa = dev.architecture
first_cz = next(locus for locus in dqa.gates["cz"].loci if all(q.startswith("QB") for q in locus))
phys_q0, phys_q1 = first_cz

# qubit_mapping tells the server which physical qubits to use.
# Our logical circuit always uses QB1/QB2; the mapping redirects to the target pair.
qubit_map = {"QB1": phys_q0, "QB2": phys_q1}
print(f"Targeting physical qubits: {qubit_map}")

# use_connectivity=False skips PL routing — we are already targeting
# a known-connected pair, so no SWAP insertion is needed.
dev_mapped = IQMDevice(server_url=SERVER_URL, wires=2, use_connectivity=False, qubit_mapping=qubit_map)


@qml.qnode(dev_mapped, shots=N_SHOTS)
def bell_mapped():
	qml.Hadamard(wires=0)
	qml.CNOT(wires=[0, 1])
	return qml.sample(wires=[0, 1])


s = bell_mapped()
bitstrings, counts = np.unique(s, axis=0, return_counts=True)
print(f"Bell state on physical qubits {phys_q0}-{phys_q1}:")
for bs, c in sorted(zip([tuple(b) for b in bitstrings], counts), key=lambda x: -x[1]):
	print(f"  |{''.join(map(str, bs))}\u27e9  {c:5d}  ({100 * c / N_SHOTS:.1f}%)")

Targeting physical qubits: {'QB1': 'QB1', 'QB2': 'QB2'}


Bell state on physical qubits QB1-QB2:
  |00⟩    526  (51.4%)
  |11⟩    498  (48.6%)


### 7e. Calibration quality metrics

`get_quality_metric_set()` returns the current gate fidelities and coherence times
from the active calibration set — useful for choosing qubits or filtering results.


In [13]:
qms = dev.client.get_quality_metric_set()
print("Observation set ID:", qms.observation_set_id)
print("Created           :", qms.created_timestamp)
print("Observation count :", len(qms.observations))

# Each observation is a calibration measurement (T1, T2, gate fidelity, etc.)
print("\nSample metrics:")
for obs in qms.observations[:10]:
	print(f"  {obs.dut_field:<45} {obs.value!r}  {obs.unit}")

Observation set ID: 00267b50-7e11-4e90-b0d1-72a7036d962f
Created           : 2026-06-16 08:42:11.654203+00:00
Observation count : 720

Sample metrics:
  metrics.ssro.measure.constant.QB1.fidelity    0.96  
  metrics.ssro.measure.constant.QB1.error_0_to_1 0.02  
  metrics.ssro.measure.constant.QB1.error_1_to_0 0.02  
  metrics.ssro.measure.constant.QB2.fidelity    0.96  
  metrics.ssro.measure.constant.QB2.error_0_to_1 0.02  
  metrics.ssro.measure.constant.QB2.error_1_to_0 0.02  
  metrics.ssro.measure.constant.QB3.fidelity    0.96  
  metrics.ssro.measure.constant.QB3.error_0_to_1 0.02  
  metrics.ssro.measure.constant.QB3.error_1_to_0 0.02  
  metrics.ssro.measure.constant.QB4.fidelity    0.96  
